In [ ]:
import sys
from pathlib import Path

# Add parent directory to Python path
sys.path.insert(0, str(Path.cwd().parent))

from data.dataset import initialize_dataloaders

train_loader, val_loader, test_loader, ans2idx, idx2ans = initialize_dataloaders()  

d:\amir lab\RS-Vamba-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: C:\Users\Nabil\.cache\kagglehub\datasets\alienxc137\earthvqa-semantic-segmentation-visual-question-ans\versions\1
Train size: 79349
Val size:   51481
Test size:  37929


In [2]:
# import os
# import shutil
# from pathlib import Path

# # Define the directory
# CHECKPOINTS_DIR = Path("./outputs/text_encoded/checkpoints")

# # Remove all .pt and .tmp files to start fresh
# if CHECKPOINTS_DIR.exists():
#     files_deleted = 0
#     for file in CHECKPOINTS_DIR.glob("*"):
#         if file.suffix in [".pt", ".tmp"]:
#             file.unlink()
#             files_deleted += 1
#     print(f"Successfully removed {files_deleted} old checkpoints. Directory is clean.")
# else:
#     print("Checkpoint directory does not exist yet. Nothing to remove.")

In [3]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from transformers import RobertaModel
# from tqdm import tqdm
# from pathlib import Path
# import gc
# import os

# os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# TEXT_DIR        = Path("./outputs/text_encoded")
# CHECKPOINTS_DIR = TEXT_DIR / "checkpoints"
# EMBEDDINGS_DIR  = TEXT_DIR / "embeddings"
# for d in [TEXT_DIR, CHECKPOINTS_DIR, EMBEDDINGS_DIR]:
#     d.mkdir(parents=True, exist_ok=True)


# class RoBERTaTextEncoder(nn.Module):
#     def __init__(self, freeze_layers=18, output_dim=768):
#         super().__init__()
#         self.roberta = RobertaModel.from_pretrained('roberta-large', add_pooling_layer=False)
#         for i, layer in enumerate(self.roberta.encoder.layer):
#             if i < freeze_layers:
#                 for param in layer.parameters():
#                     param.requires_grad = False
#         self.proj = nn.Linear(1024, output_dim)
#         self.norm = nn.LayerNorm(output_dim)

#     def forward(self, input_ids, attention_mask, return_tokens=False):
#         outputs   = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
#         hidden    = outputs.last_hidden_state
#         cls_embed = self.norm(self.proj(hidden[:, 0, :]))
#         if return_tokens:
#             token_embed = self.norm(self.proj(hidden))
#             return cls_embed, token_embed
#         return cls_embed, None


# class RoBERTaClassifier(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.encoder = RoBERTaTextEncoder(freeze_layers=18)
#         self.head    = nn.Sequential(
#             nn.Linear(768, 512),
#             nn.LayerNorm(512),
#             nn.GELU(),
#             nn.Dropout(0.2),
#             nn.Linear(512, num_classes)
#         )
#         for m in self.head.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.trunc_normal_(m.weight, std=0.02)
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)

#     def forward(self, input_ids, attention_mask, return_tokens=False):
#         cls_embed, token_embed = self.encoder(input_ids, attention_mask, return_tokens=return_tokens)
#         logits                 = self.head(cls_embed)
#         return logits, cls_embed, token_embed


# def find_latest_checkpoint(ckpt_dir: Path):
#     checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"))
#     return checkpoints[-1] if checkpoints else None


# gc.collect()
# torch.cuda.empty_cache()

# NUM_EPOCHS = 10
# device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# scaler     = torch.amp.GradScaler('cuda')
# criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)

# model_text = RoBERTaClassifier(num_classes=len(ans2idx)).to(device)
# optimizer  = optim.AdamW([
#     {'params': model_text.encoder.parameters(), 'lr': 2e-5},
#     {'params': model_text.head.parameters(),    'lr': 1e-3}
# ], weight_decay=0.05)
# scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
# )

# latest_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR)
# START_EPOCH = 0
# if latest_ckpt:
#     ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
#     model_text.load_state_dict(ckpt['model_state_dict'])
#     optimizer.load_state_dict(ckpt['optimizer_state_dict'])
#     START_EPOCH = ckpt['epoch']
#     print(f"Resumed from epoch {START_EPOCH} | val_loss: {ckpt['val_loss']:.4f}")
# else:
#     print("No checkpoint found. Starting from scratch.")

# total     = sum(p.numel() for p in model_text.parameters()) / 1e6
# trainable = sum(p.numel() for p in model_text.parameters() if p.requires_grad) / 1e6
# print(f"Total params:     {total:.1f}M")
# print(f"Trainable params: {trainable:.1f}M")
# print(f"Device:           {device}")
# print(f"Training:         epoch {START_EPOCH + 1} → {NUM_EPOCHS}")

# best_val_loss = float('inf')

# for epoch in range(START_EPOCH, NUM_EPOCHS):
#     model_text.train()
#     epoch_loss = 0.0
#     pbar       = tqdm(total=len(train_loader), desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS}", leave=True)

#     for batch in train_loader:
#         input_ids = batch['question_input_ids'].to(device, non_blocking=True)
#         attn_mask = batch['question_attn_mask'].to(device, non_blocking=True)
#         labels    = batch['answer_class_idx'].to(device, non_blocking=True)

#         optimizer.zero_grad(set_to_none=True)
#         with torch.amp.autocast('cuda'):
#             logits, _, _ = model_text(input_ids, attn_mask, return_tokens=False)
#             loss         = criterion(logits, labels)

#         scaler.scale(loss).backward()
#         torch.nn.utils.clip_grad_norm_(model_text.parameters(), max_norm=1.0)
#         scaler.step(optimizer)
#         scaler.update()

#         epoch_loss += loss.item()
#         pbar.update(1)
#         if pbar.n % 20 == 0:
#             pbar.set_postfix(loss=f"{loss.item():.4f}")

#     pbar.close()

#     model_text.eval()
#     val_loss = 0.0
#     correct  = 0
#     total_n  = 0
#     with torch.no_grad():
#         for batch in val_loader:
#             input_ids = batch['question_input_ids'].to(device, non_blocking=True)
#             attn_mask = batch['question_attn_mask'].to(device, non_blocking=True)
#             labels    = batch['answer_class_idx'].to(device, non_blocking=True)
#             with torch.amp.autocast('cuda'):
#                 logits, _, _ = model_text(input_ids, attn_mask, return_tokens=False)
#                 v_loss       = criterion(logits, labels)
#             val_loss += v_loss.item()
#             correct  += (logits.argmax(-1) == labels).sum().item()
#             total_n  += labels.size(0)
#             del logits, v_loss
#             torch.cuda.empty_cache()

#     scheduler.step()
#     avg_train = epoch_loss / len(train_loader)
#     avg_val   = val_loss   / len(val_loader)
#     val_acc   = correct    / total_n

#     if avg_val < best_val_loss:
#         best_val_loss = avg_val
#         torch.save({
#             "epoch":                epoch + 1,
#             "model_state_dict":     model_text.state_dict(),
#             "optimizer_state_dict": optimizer.state_dict(),
#             "val_loss":             avg_val,
#         }, CHECKPOINTS_DIR / "best_model.pt")
#         print(f"  New best model saved! val_loss: {avg_val:.4f}")

#     ckpt_path = CHECKPOINTS_DIR / f"epoch_{epoch+1:02d}_loss_{avg_val:.4f}.pt"
#     torch.save({
#         "epoch":                epoch + 1,
#         "model_state_dict":     model_text.state_dict(),
#         "optimizer_state_dict": optimizer.state_dict(),
#         "val_loss":             avg_val,
#     }, ckpt_path)

#     print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | train: {avg_train:.4f} | val: {avg_val:.4f} | acc: {val_acc:.3f} | lr: {scheduler.get_last_lr()[0]:.2e}")
#     gc.collect()
#     torch.cuda.empty_cache()

# print("Training complete.")


# @torch.no_grad()
# def extract_text_embeddings(loader, split_name, chunk_size=500):
#     model_text.eval()
#     chunk     = {}
#     chunk_idx = 0

#     def save_chunk():
#         nonlocal chunk, chunk_idx
#         if chunk:
#             chunk_path = EMBEDDINGS_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
#             torch.save(chunk, chunk_path)
#             print(f"  Saved chunk {chunk_idx} ({len(chunk)} items)")
#             chunk      = {}
#             chunk_idx += 1
#             gc.collect()

#     for batch in tqdm(loader, desc=f"Extracting {split_name}"):
#         input_ids = batch['question_input_ids'].to(device, non_blocking=True)
#         attn_mask = batch['question_attn_mask'].to(device, non_blocking=True)
#         image_ids = batch['image_id']
#         answers   = batch['raw_explanation']

#         with torch.amp.autocast('cuda'):
#             cls_embed, token_embed = model_text.encoder(
#                 input_ids, attn_mask, return_tokens=True
#             )

#         cls_embed   = cls_embed.cpu().to(torch.float16)
#         token_embed = token_embed.cpu().to(torch.float16)

#         for i, img_id in enumerate(image_ids):
#             key        = f"{img_id}||{i}"
#             chunk[key] = {
#                 'image_id':    img_id,
#                 'cls_embed':   cls_embed[i],
#                 'token_embed': token_embed[i],
#                 'answer':      answers[i],
#             }
#         if len(chunk) >= chunk_size:
#             save_chunk()

#         del cls_embed, token_embed
#         torch.cuda.empty_cache()

#     save_chunk()
#     print(f"Done: {split_name} → {chunk_idx} chunks saved to {EMBEDDINGS_DIR}")


# for split_name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
#     first_chunk = EMBEDDINGS_DIR / f"{split_name}_chunk_0000.pt"
#     if first_chunk.exists():
#         print(f"Skipping {split_name} — already extracted")
#     else:
#         extract_text_embeddings(loader, split_name)

# torch.save({
#     "model_state_dict": model_text.state_dict(),
#     "num_classes":      len(ans2idx),
#     "ans2idx":          ans2idx,
#     "idx2ans":          idx2ans,
#     "embed_dim":        768,
#     "backbone":         "roberta-large",
# }, TEXT_DIR / "roberta_final_model.pt")
# print("Final model saved →", TEXT_DIR / "roberta_final_model.pt")

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import RobertaModel
from tqdm import tqdm
from pathlib import Path
import gc
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

TEXT_DIR        = Path("./outputs/text_encoded")
CHECKPOINTS_DIR = TEXT_DIR / "checkpoints"
EMBEDDINGS_DIR  = TEXT_DIR / "embeddings"
for d in [TEXT_DIR, CHECKPOINTS_DIR, EMBEDDINGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


class RoBERTaTextEncoder(nn.Module):
    def __init__(self, freeze_layers=18, output_dim=768):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-large', add_pooling_layer=False)
        for i, layer in enumerate(self.roberta.encoder.layer):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        self.proj = nn.Linear(1024, output_dim)
        self.norm = nn.LayerNorm(output_dim)

    def forward(self, input_ids, attention_mask, return_tokens=False):
        outputs   = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden    = outputs.last_hidden_state
        cls_embed = self.norm(self.proj(hidden[:, 0, :]))
        if return_tokens:
            token_embed = self.norm(self.proj(hidden))
            return cls_embed, token_embed
        return cls_embed, None


class RoBERTaClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = RoBERTaTextEncoder(freeze_layers=18)
        self.head    = nn.Sequential(
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, input_ids, attention_mask, return_tokens=False):
        cls_embed, token_embed = self.encoder(input_ids, attention_mask, return_tokens=return_tokens)
        logits                 = self.head(cls_embed)
        return logits, cls_embed, token_embed


def find_latest_checkpoint(ckpt_dir: Path):
    checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"))
    return checkpoints[-1] if checkpoints else None


gc.collect()
torch.cuda.empty_cache()

NUM_EPOCHS = 3
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scaler     = torch.amp.GradScaler('cuda')
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)

model_text = RoBERTaClassifier(num_classes=len(ans2idx)).to(device)
optimizer  = optim.AdamW([
    {'params': model_text.encoder.parameters(), 'lr': 2e-5},
    {'params': model_text.head.parameters(),    'lr': 1e-3}
], weight_decay=0.05)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

latest_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR)
START_EPOCH = 0
if latest_ckpt:
    ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
    model_text.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    START_EPOCH = ckpt['epoch']
    print(f"Resumed from epoch {START_EPOCH} | val_loss: {ckpt['val_loss']:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

total     = sum(p.numel() for p in model_text.parameters()) / 1e6
trainable = sum(p.numel() for p in model_text.parameters() if p.requires_grad) / 1e6
print(f"Total params:     {total:.1f}M")
print(f"Trainable params: {trainable:.1f}M")
print(f"Device:           {device}")
print(f"Training:         epoch {START_EPOCH + 1} → {NUM_EPOCHS}")

best_val_loss = float('inf')

def keep_top_k_checkpoints(ckpt_dir: Path, k: int = 3):
    checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"), key=lambda p: float(p.stem.split("_loss_")[-1]))
    for ckpt in checkpoints[k:]:
        ckpt.unlink()
        print(f"  Removed old checkpoint: {ckpt.name}")

for epoch in range(START_EPOCH, NUM_EPOCHS):
    model_text.train()
    epoch_loss = 0.0
    pbar       = tqdm(total=len(train_loader), desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS}", leave=True)

    for batch in train_loader:
        input_ids = batch['question_input_ids'].to(device, non_blocking=True)
        attn_mask = batch['question_attn_mask'].to(device, non_blocking=True)
        labels    = batch['answer_class_idx'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits, _, _ = model_text(input_ids, attn_mask, return_tokens=False)
            loss         = criterion(logits, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model_text.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.update(1)
        if pbar.n % 20 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    pbar.close()

    model_text.eval()
    val_loss = 0.0
    correct  = 0
    total_n  = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['question_input_ids'].to(device, non_blocking=True)
            attn_mask = batch['question_attn_mask'].to(device, non_blocking=True)
            labels    = batch['answer_class_idx'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                logits, _, _ = model_text(input_ids, attn_mask, return_tokens=False)
                v_loss       = criterion(logits, labels)
            val_loss += v_loss.item()
            correct  += (logits.argmax(-1) == labels).sum().item()
            total_n  += labels.size(0)
            del logits, v_loss
            torch.cuda.empty_cache()

    scheduler.step()
    avg_train = epoch_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    val_acc   = correct    / total_n

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model_text.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss":             avg_val,
        }, CHECKPOINTS_DIR / "best_model.pt")
        print(f"  New best model saved! val_loss: {avg_val:.4f}")

    ckpt_path = CHECKPOINTS_DIR / f"epoch_{epoch+1:02d}_loss_{avg_val:.4f}.pt"
    torch.save({
        "epoch":                epoch + 1,
        "model_state_dict":     model_text.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss":             avg_val,
    }, ckpt_path)
    keep_top_k_checkpoints(CHECKPOINTS_DIR, k=3)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | train: {avg_train:.4f} | val: {avg_val:.4f} | acc: {val_acc:.3f} | lr: {scheduler.get_last_lr()[0]:.2e}")
    gc.collect()
    torch.cuda.empty_cache()

print("Training complete.")


@torch.no_grad()
def extract_text_embeddings(loader, split_name, chunk_size=1000):
    model_text.eval()
    chunk      = {}
    chunk_idx  = 0
    global_idx = 0

    def save_chunk():
        nonlocal chunk, chunk_idx
        if chunk:
            chunk_path = EMBEDDINGS_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
            torch.save(chunk, chunk_path)
            print(f"  Saved chunk {chunk_idx} ({len(chunk)} items)")
            chunk      = {}
            chunk_idx += 1
            gc.collect()

    for batch in tqdm(loader, desc=f"Extracting {split_name}"):
        input_ids    = batch['question_input_ids'].to(device, non_blocking=True)
        attn_mask    = batch['question_attn_mask'].to(device, non_blocking=True)
        image_ids    = batch['image_id']
        answers      = batch['raw_explanation']
        attn_mask_cpu = attn_mask.cpu()

        with torch.amp.autocast('cuda'):
            cls_embed, token_embed = model_text.encoder(
                input_ids, attn_mask, return_tokens=True
            )

        cls_embed   = cls_embed.cpu().to(torch.float16)
        token_embed = token_embed.cpu().to(torch.float16)

        for i, img_id in enumerate(image_ids):
            valid_len  = int(attn_mask_cpu[i].sum().item())
            key        = f"{img_id}||{global_idx}"
            global_idx += 1
            chunk[key] = {
                'image_id':    img_id,
                'cls_embed':   cls_embed[i],
                'token_embed': token_embed[i, :valid_len, :],
                'seq_len':     valid_len,
                'answer':      answers[i],
            }

        if len(chunk) >= chunk_size:
            save_chunk()

        del cls_embed, token_embed, attn_mask_cpu
        torch.cuda.empty_cache()

    save_chunk()
    print(f"Done: {split_name} → {chunk_idx} chunks saved to {EMBEDDINGS_DIR}")


def refresh_embeddings(embeddings_dir: Path, split_name: str):
    existing = list(embeddings_dir.glob(f"{split_name}_chunk_*.pt"))
    if existing:
        print(f"Deleting {len(existing)} old {split_name} chunks...")
        for f in existing:
            f.unlink()
        print(f"  Done — {split_name} cleared")

for split_name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    refresh_embeddings(EMBEDDINGS_DIR, split_name)
    extract_text_embeddings(loader, split_name)

torch.save({
    "model_state_dict": model_text.state_dict(),
    "num_classes":      len(ans2idx),
    "ans2idx":          ans2idx,
    "idx2ans":          idx2ans,
    "embed_dim":        768,
    "backbone":         "roberta-large",
}, TEXT_DIR / "roberta_final_model.pt")
print("Final model saved →", TEXT_DIR / "roberta_final_model.pt")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4991.52it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No checkpoint found. Starting from scratch.
Total params:     355.6M
Trainable params: 128.8M
Device:           cuda
Training:         epoch 1 → 3


Epoch 01/3: 100%|██████████| 2480/2480 [14:34<00:00,  2.83it/s, loss=1.5397]


  New best model saved! val_loss: 1.5798
Epoch 01/3 | train: 1.6388 | val: 1.5798 | acc: 0.649 | lr: 1.50e-05


Epoch 02/3: 100%|██████████| 2480/2480 [14:19<00:00,  2.89it/s, loss=1.6463]


...

  Saved chunk 35 (1024 items)


Extracting test: 100%|█████████▉| 1184/1186 [06:53<00:00,  3.55it/s]

  Saved chunk 36 (1024 items)


Extracting test: 100%|██████████| 1186/1186 [06:53<00:00,  2.87it/s]


  Saved chunk 37 (41 items)
Done: test → 38 chunks saved to outputs\text_encoded\embeddings
Final model saved → outputs\text_encoded\roberta_final_model.pt


In [5]:
from pathlib import Path

for base, name in [
    ("./outputs/vit_encoded/embeddings", "ViT"),
    ("./outputs/text_encoded/embeddings", "RoBERTa")
]:
    p = Path(base)
    print(f"\n{name} embeddings:")
    for split in ["train", "val", "test"]:
        chunks = sorted(p.glob(f"{split}_chunk_*.pt"))
        size_mb = sum(c.stat().st_size for c in chunks) / 1e6
        print(f"  {split}: {len(chunks)} chunks | {size_mb:.1f} MB")


ViT embeddings:
  train: 0 chunks | 0.0 MB
  val: 0 chunks | 0.0 MB
  test: 0 chunks | 0.0 MB

RoBERTa embeddings:
  train: 78 chunks | 7940.9 MB
  val: 51 chunks | 5152.0 MB
  test: 38 chunks | 3795.2 MB
